In [1]:
# ============================================================
# Cell 1: Imports
# ============================================================
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             brier_score_loss, confusion_matrix, fbeta_score)
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve, precision_recall_curve
from pytorch_tabnet.tab_model import TabNetClassifier
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import shap
import joblib
import json
import os
import time
import warnings
warnings.filterwarnings('ignore')

# Seed
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("✅ Libraries imported successfully!")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {device}")

✅ Libraries imported successfully!
PyTorch : 2.5.1+cu121
CUDA    : True
GPU     : NVIDIA GeForce RTX 3050 Ti Laptop GPU
Device  : cuda


In [2]:
# ============================================================
# Cell 2: Paths & Constants
# ============================================================

DATA_DIR    = "data"
MODELS_DIR  = "models"
RESULTS_DIR = "results"
PLOTS_DIR   = "plots"

TRAIN_PATH = os.path.join(DATA_DIR, "train_forecast24_undersampled005_patientaware.csv")
VAL_PATH   = os.path.join(DATA_DIR, "val_forecast24.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test_forecast24.csv")

# نفس الـ 13 features المعتمدة
FEATURE_COLS = [
    'mean_hr', 'mean_spo2', 'sd_hr', 'sd_spo2',
    'skewness_hr', 'skewness_spo2', 'kurtosis_hr', 'kurtosis_spo2',
    'max_xc_hr_spo2', 'min_xc_hr_spo2',
    'sub', 'sepsis_window', 'blackout_window'
]

TARGET_COL  = 'y_forecast_24h'
PATIENT_COL = 'new_id'
TIME_COL    = 'seconds_since_birth'

# Hyperparameters
BATCH_SIZE = 256
EPOCHS     = 50
LR         = 1e-3

print("✅ Paths & Constants defined!")
print(f"Features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

✅ Paths & Constants defined!
Features (13): ['mean_hr', 'mean_spo2', 'sd_hr', 'sd_spo2', 'skewness_hr', 'skewness_spo2', 'kurtosis_hr', 'kurtosis_spo2', 'max_xc_hr_spo2', 'min_xc_hr_spo2', 'sub', 'sepsis_window', 'blackout_window']


In [3]:
# ============================================================
# Cell 3: Load Data & Preprocessing
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("✅ Data loaded!")
print(f"Train : {train_df.shape}")
print(f"Val   : {val_df.shape}")
print(f"Test  : {test_df.shape}")

# Features & Labels
X_train = train_df[FEATURE_COLS].values.astype(np.float32)
y_train = train_df[TARGET_COL].values.astype(int)
train_patients = train_df[PATIENT_COL].values
train_times    = train_df[TIME_COL].values

X_val = val_df[FEATURE_COLS].values.astype(np.float32)
y_val = val_df[TARGET_COL].values.astype(int)
val_patients = val_df[PATIENT_COL].values
val_times    = val_df[TIME_COL].values

X_test = test_df[FEATURE_COLS].values.astype(np.float32)
y_test = test_df[TARGET_COL].values.astype(int)
test_patients = test_df[PATIENT_COL].values
test_times    = test_df[TIME_COL].values

# Scaler
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(MODELS_DIR, "tabnet_scaler.pkl"))

# Class weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
pos_weight = neg / pos

print(f"\n✅ Preprocessing done!")
print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"pos_weight: {pos_weight:.4f}")
print(f"\n📊 Class Distribution:")
print(f"Train positive: {y_train.mean():.4f}")
print(f"Val   positive: {y_val.mean():.4f}")
print(f"Test  positive: {y_test.mean():.4f}")

✅ Data loaded!
Train : (45717, 19)
Val   : (769979, 18)
Test  : (617005, 18)

✅ Preprocessing done!
X_train: (45717, 13) | X_val: (769979, 13) | X_test: (617005, 13)
pos_weight: 0.5149

📊 Class Distribution:
Train positive: 0.6601
Val   positive: 0.0075
Test  positive: 0.0125


In [4]:
# ============================================================
# Cell 4: TabNet Model & Training
# ============================================================

# Class weights للـ TabNet
class_weight = {0: 1.0, 1: pos_weight}

# TabNet Model
tabnet_model = TabNetClassifier(
    n_d             = 32,        # عرض الـ decision step
    n_a             = 32,        # عرض الـ attention
    n_steps         = 5,         # عدد الـ steps
    gamma           = 1.5,       # معامل الـ sparsity
    n_independent   = 2,
    n_shared        = 2,
    lambda_sparse   = 1e-4,      # L1 على الـ feature selection
    optimizer_fn    = torch.optim.AdamW,
    optimizer_params= {'lr': LR, 'weight_decay': 1e-4},
    scheduler_fn    = torch.optim.lr_scheduler.ReduceLROnPlateau,
    scheduler_params= {
        'mode'    : 'max',
        'factor'  : 0.5,
        'patience': 5
    },
    mask_type       = 'sparsemax',
    device_name     = 'cuda' if torch.cuda.is_available() else 'cpu',
    verbose         = 1,
    seed            = SEED
)

print("✅ TabNet Model created!")
print(f"\n🚀 Training started...")
training_start = time.time()

tabnet_model.fit(
    X_train        = X_train,
    y_train        = y_train,
    eval_set       = [(X_val, y_val)],
    eval_name      = ['val'],
    eval_metric    = ['auc'],
    max_epochs     = EPOCHS,
    patience       = 10,
    batch_size     = BATCH_SIZE,
    virtual_batch_size = 128,
    num_workers    = 0,
    drop_last      = False,
    weights        = class_weight
)

total_time = time.time() - training_start

print(f"\n✅ Training finished!")
print(f"Total time : {total_time/60:.2f} minutes")

# حفظ النموذج
tabnet_model.save_model(os.path.join(MODELS_DIR, "tabnet_best"))
print(f"Model saved ✅")

✅ TabNet Model created!

🚀 Training started...
epoch 0  | loss: 0.66965 | val_auc: 0.78238 |  0:00:56s
epoch 1  | loss: 0.50372 | val_auc: 0.80529 |  0:01:52s
epoch 2  | loss: 0.4734  | val_auc: 0.81481 |  0:02:47s
epoch 3  | loss: 0.45523 | val_auc: 0.82403 |  0:03:43s
epoch 4  | loss: 0.44315 | val_auc: 0.83693 |  0:04:39s
epoch 5  | loss: 0.43506 | val_auc: 0.83158 |  0:05:34s
epoch 6  | loss: 0.43025 | val_auc: 0.8351  |  0:06:30s
epoch 7  | loss: 0.42683 | val_auc: 0.83176 |  0:07:27s
epoch 8  | loss: 0.42599 | val_auc: 0.83195 |  0:08:22s
epoch 9  | loss: 0.42767 | val_auc: 0.83256 |  0:09:20s
epoch 10 | loss: 0.42144 | val_auc: 0.83857 |  0:10:18s
epoch 11 | loss: 0.42099 | val_auc: 0.83623 |  0:11:15s
epoch 12 | loss: 0.41921 | val_auc: 0.83825 |  0:12:11s
epoch 13 | loss: 0.41697 | val_auc: 0.83819 |  0:13:06s
epoch 14 | loss: 0.41312 | val_auc: 0.83846 |  0:14:01s
epoch 15 | loss: 0.41306 | val_auc: 0.83971 |  0:14:57s
epoch 16 | loss: 0.41152 | val_auc: 0.83885 |  0:15:52s
e

In [5]:
# ============================================================
# Cell 5: Evaluate
# ============================================================

print("✅ Getting predictions...")

# Val predictions
val_probs_raw  = tabnet_model.predict_proba(X_val)[:, 1]
test_probs_raw = tabnet_model.predict_proba(X_test)[:, 1]

val_roc  = roc_auc_score(y_val, val_probs_raw)
val_pr   = average_precision_score(y_val, val_probs_raw)
test_roc = roc_auc_score(y_test, test_probs_raw)
test_pr  = average_precision_score(y_test, test_probs_raw)

print(f"\n📊 Val  → ROC-AUC: {val_roc:.4f} | PR-AUC: {val_pr:.4f}")
print(f"📊 Test → ROC-AUC: {test_roc:.4f} | PR-AUC: {test_pr:.4f}")

✅ Getting predictions...

📊 Val  → ROC-AUC: 0.8438 | PR-AUC: 0.5372
📊 Test → ROC-AUC: 0.8568 | PR-AUC: 0.5327


In [6]:
# ============================================================
# Cell 6: Calibration & Threshold
# ============================================================

# Platt Scaling
platt = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
platt.fit(val_probs_raw.reshape(-1, 1), y_val)

val_probs_cal  = platt.predict_proba(val_probs_raw.reshape(-1, 1))[:, 1]
test_probs_cal = platt.predict_proba(test_probs_raw.reshape(-1, 1))[:, 1]

joblib.dump(platt, os.path.join(MODELS_DIR, "tabnet_calibration.pkl"))

brier_before = brier_score_loss(y_val, val_probs_raw)
brier_after  = brier_score_loss(y_val, val_probs_cal)

print(f"✅ Calibration done!")
print(f"Brier Before : {brier_before:.4f}")
print(f"Brier After  : {brier_after:.4f}")

# Threshold على F2-score
thresholds     = np.arange(0.001, 0.5, 0.001)
best_threshold = 0.5
best_f2        = 0
f2_scores      = []

for thresh in thresholds:
    preds = (val_probs_cal >= thresh).astype(int)
    f2    = fbeta_score(y_val, preds, beta=2, zero_division=0)
    f2_scores.append(f2)
    if f2 > best_f2:
        best_f2        = f2
        best_threshold = thresh

print(f"\n✅ Threshold selected!")
print(f"Best Threshold : {best_threshold:.3f}")
print(f"Best F2-score  : {best_f2:.4f}")

# حفظ
cal_params = {
    "coef"      : platt.coef_[0][0],
    "intercept" : platt.intercept_[0],
    "method"    : "Platt Scaling"
}
with open(os.path.join(RESULTS_DIR, "tabnet_calibration.json"), "w") as f:
    json.dump(cal_params, f, indent=4)

threshold_info = {
    "best_threshold" : float(best_threshold),
    "best_f2"        : float(best_f2),
    "metric"         : "F2-score",
    "selected_on"    : "validation set"
}
with open(os.path.join(RESULTS_DIR, "tabnet_threshold.json"), "w") as f:
    json.dump(threshold_info, f, indent=4)

# Calibration Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, probs, title in zip(
    axes,
    [val_probs_raw, val_probs_cal],
    ["Before Calibration", "After Calibration (Platt)"]):
    fraction_pos, mean_pred = calibration_curve(
        y_val, probs, n_bins=10, strategy='uniform')
    ax.plot(mean_pred, fraction_pos, 'o-', color='green', label='TabNet')
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Fraction of Positives')
    ax.set_title(f'TabNet — {title}')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "tabnet_calibration.png"), dpi=150)
plt.show()
print("Plot saved ✅")

✅ Calibration done!
Brier Before : 0.1282
Brier After  : 0.0048

✅ Threshold selected!
Best Threshold : 0.423
Best F2-score  : 0.5686
Plot saved ✅


In [7]:
# ============================================================
# Cell 7: Window-level Evaluation
# ============================================================

test_preds_cal = (test_probs_cal >= best_threshold).astype(int)
val_preds_cal  = (val_probs_cal  >= best_threshold).astype(int)

acc       = accuracy_score(y_test, test_preds_cal)
precision = precision_score(y_test, test_preds_cal, zero_division=0)
recall    = recall_score(y_test, test_preds_cal, zero_division=0)
f1        = f1_score(y_test, test_preds_cal, zero_division=0)
f2        = fbeta_score(y_test, test_preds_cal, beta=2, zero_division=0)
roc_auc   = roc_auc_score(y_test, test_probs_cal)
pr_auc    = average_precision_score(y_test, test_probs_cal)
brier     = brier_score_loss(y_test, test_probs_cal)
tn, fp, fn, tp = confusion_matrix(y_test, test_preds_cal).ravel()
specificity = tn / (tn + fp)
ppv         = tp / (tp + fp) if (tp + fp) > 0 else 0
npv         = tn / (tn + fn) if (tn + fn) > 0 else 0

print("=" * 55)
print("   TabNet — Window-level Evaluation (Test)")
print("=" * 55)
print(f"  Accuracy    : {acc:.4f}")
print(f"  Precision   : {precision:.4f}")
print(f"  Recall      : {recall:.4f}")
print(f"  Specificity : {specificity:.4f}")
print(f"  F1-score    : {f1:.4f}")
print(f"  F2-score    : {f2:.4f}")
print(f"  PPV         : {ppv:.4f}")
print(f"  NPV         : {npv:.4f}")
print(f"  ROC-AUC     : {roc_auc:.4f}")
print(f"  PR-AUC      : {pr_auc:.4f}")
print(f"  Brier Score : {brier:.4f}")
print("=" * 55)
print(f"\n  Confusion Matrix:")
print(f"  TN={tn:,}  FP={fp:,}")
print(f"  FN={fn:,}   TP={tp:,}")

# حفظ
window_metrics = {
    "level"       : "window",
    "threshold"   : float(best_threshold),
    "accuracy"    : acc, "precision"   : precision,
    "recall"      : recall, "specificity" : specificity,
    "f1"          : f1, "f2"           : f2,
    "ppv"         : ppv, "npv"          : npv,
    "roc_auc"     : roc_auc, "pr_auc"  : pr_auc,
    "brier"       : brier,
    "tp": int(tp), "tn": int(tn),
    "fp": int(fp), "fn": int(fn)
}
with open(os.path.join(RESULTS_DIR, "tabnet_window_metrics.json"), "w") as f:
    json.dump(window_metrics, f, indent=4)

# Confusion Matrix Plot
plt.figure(figsize=(6, 5))
cm = np.array([[tn, fp], [fn, tp]])
sns.heatmap(cm, annot=True, fmt=',', cmap='Greens',
            xticklabels=['Pred 0', 'Pred 1'],
            yticklabels=['True 0', 'True 1'])
plt.title('TabNet — Confusion Matrix (Window-level)')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "tabnet_confusion_matrix.png"), dpi=150)
plt.show()
print("\n✅ Window-level metrics saved!")

   TabNet — Window-level Evaluation (Test)
  Accuracy    : 0.9936
  Precision   : 0.9795
  Recall      : 0.4974
  Specificity : 0.9999
  F1-score    : 0.6598
  F2-score    : 0.5517
  PPV         : 0.9795
  NPV         : 0.9937
  ROC-AUC     : 0.8568
  PR-AUC      : 0.5327
  Brier Score : 0.0079

  Confusion Matrix:
  TN=609,243  FP=80
  FN=3,861   TP=3,821

✅ Window-level metrics saved!


In [8]:
# ============================================================
# Cell 8: Save Predictions & Patient-level
# ============================================================

val_pred_df = pd.DataFrame({
    'patient_id'         : val_df[PATIENT_COL].values,
    'seconds_since_birth': val_df[TIME_COL].values,
    'y_true'             : y_val,
    'y_prob'             : val_probs_raw,
    'y_prob_cal'         : val_probs_cal,
    'y_pred'             : val_preds_cal
})

test_pred_df = pd.DataFrame({
    'patient_id'         : test_df[PATIENT_COL].values,
    'seconds_since_birth': test_df[TIME_COL].values,
    'y_true'             : y_test,
    'y_prob'             : test_probs_raw,
    'y_prob_cal'         : test_probs_cal,
    'y_pred'             : test_preds_cal
})

val_pred_df.to_csv(
    os.path.join(RESULTS_DIR, "tabnet_val_predictions_patientwise.csv"), index=False)
test_pred_df.to_csv(
    os.path.join(RESULTS_DIR, "tabnet_test_predictions_patientwise.csv"), index=False)

print("✅ Predictions saved!")

# Patient-level
sepsis_detected = 0
sepsis_missed   = 0
false_alarm     = 0

for pid, group in test_pred_df.groupby('patient_id'):
    y_true_p = group['y_true'].max()
    y_pred_p = group['y_pred'].max()

    if y_true_p == 1 and y_pred_p == 1:
        sepsis_detected += 1
    elif y_true_p == 1 and y_pred_p == 0:
        sepsis_missed += 1
    elif y_true_p == 0 and y_pred_p == 1:
        false_alarm += 1

print(f"\n{'='*50}")
print(f"   TabNet — Patient-level Summary (Test)")
print(f"{'='*50}")
print(f"  ✅ مرضى Sepsis مكتشفين  : {sepsis_detected}/22")
print(f"  ❌ مرضى Sepsis ضاعوا    : {sepsis_missed}/22")
print(f"  ⚠️  إنذارات كاذبة        : {false_alarm}/44")
print(f"{'='*50}")

# حفظ
patient_metrics = {
    "level"          : "patient",
    "total_patients" : 66,
    "sepsis"         : 22,
    "normal"         : 44,
    "detected"       : sepsis_detected,
    "missed"         : sepsis_missed,
    "false_alarm"    : false_alarm,
    "sensitivity"    : sepsis_detected / 22,
    "specificity"    : (44 - false_alarm) / 44
}
with open(os.path.join(RESULTS_DIR, "tabnet_patient_metrics_test.json"), "w") as f:
    json.dump(patient_metrics, f, indent=4)

print("\n✅ Patient-level saved!")

✅ Predictions saved!

   TabNet — Patient-level Summary (Test)
  ✅ مرضى Sepsis مكتشفين  : 22/22
  ❌ مرضى Sepsis ضاعوا    : 0/22
  ⚠️  إنذارات كاذبة        : 10/44

✅ Patient-level saved!


In [9]:
# ============================================================
# Cell 9: ROC, PR & Plots
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
fpr, tpr, _ = roc_curve(y_test, test_probs_cal)
axes[0].plot(fpr, tpr, color='green', linewidth=2,
             label=f'TabNet (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], '--', color='gray', label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='green')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('TabNet — ROC Curve (Test Set)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PR
precision_curve, recall_curve, _ = precision_recall_curve(y_test, test_probs_cal)
axes[1].plot(recall_curve, precision_curve, color='darkgreen', linewidth=2,
             label=f'TabNet (AUC = {pr_auc:.4f})')
axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--',
                label=f'Baseline ({y_test.mean():.4f})')
axes[1].fill_between(recall_curve, precision_curve, alpha=0.1, color='darkgreen')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('TabNet — Precision-Recall Curve (Test Set)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "tabnet_roc_pr_curves.png"), dpi=150)
plt.show()
print("✅ ROC & PR saved!")

✅ ROC & PR saved!


In [10]:
# ============================================================
# Cell 10: Feature Importance
# ============================================================

# TabNet عنده Feature Importance مدمج — ميزته الأساسية!
feature_importance = tabnet_model.feature_importances_

importance_df = pd.DataFrame({
    'feature'    : FEATURE_COLS,
    'importance' : feature_importance
}).sort_values('importance', ascending=False)

print("📊 TabNet Feature Importance (Built-in):")
print(importance_df.to_string(index=False))

# رسمة
plt.figure(figsize=(10, 6))
colors = ['#d32f2f' if x > 0.15 else '#f57c00' if x > 0.05 else '#388e3c'
          for x in importance_df['importance']]
plt.barh(importance_df['feature'], importance_df['importance'], color=colors)
plt.xlabel('Importance Score')
plt.title('TabNet — Built-in Feature Importance')
plt.axvline(x=0.15, color='red', linestyle='--', alpha=0.5, label='High')
plt.axvline(x=0.05, color='orange', linestyle='--', alpha=0.5, label='Medium')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "tabnet_feature_importance.png"), dpi=150)
plt.show()

importance_df.to_csv(
    os.path.join(RESULTS_DIR, "tabnet_feature_importance.csv"), index=False)

# Permutation Importance
print("\n🔄 Computing Permutation Importance...")
print(f"{'Feature':<25} {'PR-AUC':<10} {'Drop':<10} {'Importance'}")
print("-" * 60)

baseline_prauc = average_precision_score(y_val, 
    tabnet_model.predict_proba(X_val)[:, 1])

perm_results = []
for i, feature in enumerate(FEATURE_COLS):
    X_val_perm = X_val.copy()
    np.random.shuffle(X_val_perm[:, i])
    perm_probs = tabnet_model.predict_proba(X_val_perm)[:, 1]
    perm_prauc = average_precision_score(y_val, perm_probs)
    drop       = baseline_prauc - perm_prauc

    perm_results.append({
        'feature'    : feature,
        'prauc'      : perm_prauc,
        'drop'       : drop,
        'importance' : max(drop, 0)
    })
    print(f"{feature:<25} {perm_prauc:<10.4f} {drop:<10.4f} "
          f"{'🔴 مهمة جداً' if drop > 0.05 else '🟡 متوسطة' if drop > 0.01 else '🟢 ضعيفة'}")

perm_df = pd.DataFrame(perm_results).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#d32f2f' if x > 0.05 else '#f57c00' if x > 0.01 else '#388e3c'
          for x in perm_df['importance']]
plt.barh(perm_df['feature'], perm_df['importance'], color=colors)
plt.xlabel('Importance (Drop in PR-AUC)')
plt.title('TabNet — Permutation Feature Importance')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "tabnet_permutation_importance.png"), dpi=150)
plt.show()

perm_df.to_csv(
    os.path.join(RESULTS_DIR, "tabnet_permutation_importance.csv"), index=False)
print("\n✅ Feature importance saved!")

📊 TabNet Feature Importance (Built-in):
        feature  importance
  sepsis_window    0.368352
    skewness_hr    0.091249
          sd_hr    0.085952
      mean_spo2    0.080593
        mean_hr    0.071093
 max_xc_hr_spo2    0.070000
  skewness_spo2    0.051373
            sub    0.041241
        sd_spo2    0.038921
 min_xc_hr_spo2    0.034566
    kurtosis_hr    0.031486
blackout_window    0.017807
  kurtosis_spo2    0.017366

🔄 Computing Permutation Importance...
Feature                   PR-AUC     Drop       Importance
------------------------------------------------------------
mean_hr                   0.5343     0.0029     🟢 ضعيفة
mean_spo2                 0.5388     -0.0016    🟢 ضعيفة
sd_hr                     0.5294     0.0078     🟢 ضعيفة
sd_spo2                   0.5369     0.0003     🟢 ضعيفة
skewness_hr               0.5373     -0.0001    🟢 ضعيفة
skewness_spo2             0.5346     0.0027     🟢 ضعيفة
kurtosis_hr               0.5363     0.0010     🟢 ضعيفة
kurtosis_spo2    

In [11]:
# ============================================================
# Cell 11: Final Summary
# ============================================================

# Inference time
start = time.time()
_ = tabnet_model.predict_proba(X_test)
inference_time = time.time() - start

# Model size
model_size  = os.path.getsize(
    os.path.join(MODELS_DIR, "tabnet_best.zip")) / (1024 * 1024)
scaler_size = os.path.getsize(
    os.path.join(MODELS_DIR, "tabnet_scaler.pkl")) / 1024

# Parameters
total_params     = sum(p.numel() for p in tabnet_model.network.parameters())
trainable_params = sum(p.numel() for p in tabnet_model.network.parameters()
                       if p.requires_grad)

final_summary = {
    "model"      : "TabNet",
    "features"   : FEATURE_COLS,
    "architecture" : {
        "n_d"           : 32,
        "n_a"           : 32,
        "n_steps"       : 5,
        "gamma"         : 1.5,
        "mask_type"     : "sparsemax",
        "total_params"  : total_params,
        "trainable_params": trainable_params
    },
    "training" : {
        "optimizer"        : "AdamW",
        "loss"             : "BCEWithLogitsLoss",
        "scheduler"        : "ReduceLROnPlateau",
        "early_stopping"   : "patience=10",
        "training_time_min": round(total_time/60, 2)
    },
    "calibration" : {
        "method"       : "Platt Scaling",
        "brier_before" : float(brier_score_loss(y_val, val_probs_raw)),
        "brier_after"  : float(brier_score_loss(y_val, val_probs_cal))
    },
    "threshold" : {
        "value"   : float(best_threshold),
        "metric"  : "F2-score",
        "f2_value": float(best_f2)
    },
    "window_metrics" : {
        "accuracy"    : acc, "precision"   : precision,
        "recall"      : recall, "specificity" : specificity,
        "f1"          : f1, "f2"           : f2,
        "ppv"         : ppv, "npv"          : npv,
        "roc_auc"     : roc_auc, "pr_auc"  : pr_auc,
        "brier"       : brier,
        "tp": int(tp), "tn": int(tn),
        "fp": int(fp), "fn": int(fn)
    },
    "patient_metrics" : {
        "total"       : 66, "sepsis"     : 22,
        "normal"      : 44, "detected"   : sepsis_detected,
        "missed"      : sepsis_missed, "false_alarm": false_alarm
    },
    "complexity" : {
        "total_params"    : total_params,
        "training_min"    : round(total_time/60, 2),
        "inference_sec"   : round(inference_time, 2),
        "per_sample_ms"   : round(inference_time/len(y_test)*1000, 4),
        "model_size_mb"   : round(model_size, 2),
        "scaler_size_kb"  : round(scaler_size, 2)
    }
}

with open(os.path.join(RESULTS_DIR, "tabnet_final_summary.json"), "w") as f:
    json.dump(final_summary, f, indent=4)

print("=" * 57)
print("   TabNet — FINAL COMPLETE SUMMARY")
print("=" * 57)
print(f"\n🧠 Architecture:")
print(f"  Parameters      : {total_params:,}")
print(f"  n_d=n_a         : 32")
print(f"  n_steps         : 5")
print(f"  mask_type       : sparsemax")
print(f"\n📊 Window-level (Test):")
print(f"  ROC-AUC         : {roc_auc:.4f}")
print(f"  PR-AUC          : {pr_auc:.4f}")
print(f"  Recall          : {recall:.4f}")
print(f"  Precision       : {precision:.4f}")
print(f"  F2-score        : {f2:.4f}")
print(f"  FP (إنذار كاذب) : {fp:,}")
print(f"\n👥 Patient-level (Test):")
print(f"  مكتشفين         : {sepsis_detected}/22 ✅")
print(f"  ضاعوا           : {sepsis_missed}/22")
print(f"  إنذارات كاذبة   : {false_alarm}/44")
print(f"\n⏱️  Complexity:")
print(f"  Training        : {total_time/60:.2f} min")
print(f"  Inference/sample: {inference_time/len(y_test)*1000:.4f} ms")
print(f"  Model size      : {model_size:.2f} MB")
print("=" * 57)
print("\n🎉 TabNet Model Complete!")

   TabNet — FINAL COMPLETE SUMMARY

🧠 Architecture:
  Parameters      : 116,630
  n_d=n_a         : 32
  n_steps         : 5
  mask_type       : sparsemax

📊 Window-level (Test):
  ROC-AUC         : 0.8568
  PR-AUC          : 0.5327
  Recall          : 0.4974
  Precision       : 0.9795
  F2-score        : 0.5517
  FP (إنذار كاذب) : 80

👥 Patient-level (Test):
  مكتشفين         : 22/22 ✅
  ضاعوا           : 0/22
  إنذارات كاذبة   : 10/44

⏱️  Complexity:
  Training        : 31.64 min
  Inference/sample: 0.1917 ms
  Model size      : 0.45 MB

🎉 TabNet Model Complete!
